<a href="https://colab.research.google.com/github/webomurga/dsai543-final/blob/main/Model_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data Acquisition & Setup

**Dataset Versioning Update:** The original proposal was based on the preliminary preprint release figures of the CellBinDB dataset (270 H&E, 86 mIF). However, upon downloading the final, peer-reviewed version of the dataset from Zenodo for the execution phase, the repository had been officially updated by the authors. Corrupted or poorly annotated mIF images were pruned to ensure benchmark quality (reducing the count to 60), while the H&E subset was expanded with additional validated patches (increasing the count to 305). To ensure the most robust and replicable statistical analysis, this project utilized the entirety of the updated, final dataset.

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

BASE_DIR = "" # @param {"type":"string"}"

Mounted at /content/drive


In [ ]:
raw_dir = os.path.join(BASE_DIR, "data", "raw_baseline")
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "results"), exist_ok=True)

dataset_url = "https://zenodo.org/records/14312044/files/CellBinDB.zip?download=1"
local_zip = "/content/CellBinDB.zip"

if not os.path.exists(os.path.join(raw_dir, "CellBinDB")):
    print("Downloading CellBinDB...")
    !wget -q -O "{local_zip}" "{dataset_url}"
    print("Extracting to Drive...")
    !unzip -q -o "{local_zip}" -d "{raw_dir}"
    os.remove(local_zip)
    print("Setup complete!")
else:
    print("Dataset already downloaded and extracted.")

Dataset already downloaded and extracted.


#### The HED Color Deconvolution Matrix

The `hed_matrix` defined in the `EnhancementPipeline` is an industry-standard **Color Deconvolution Matrix** used in digital pathology. It is designed to mathematically separate a standard RGB histological image into distinct channels representing the physical dyes used on the tissue slide.

#### Matrix Breakdown (Stains vs. RGB Optical Density)
Each row represents a specific physical dye, and the columns define the Normalized Optical Density (OD) for Red, Green, and Blue light absorption:

* **Row 1 - Hematoxylin `[0.650, 0.704, 0.286]`**: Typically stains cell nuclei blue/purple.
* **Row 2 - Eosin `[0.072, 0.990, 0.105]`**: Typically stains the cytoplasm and extracellular matrix pink/red. Notice the high middle value (`0.990`), indicating strong green light absorption.
* **Row 3 - DAB `[0.268, 0.570, 0.776]`**: 3,3'-Diaminobenzidine, a brown stain frequently used in Immunohistochemistry (IHC) to highlight specific proteins.

#### Origin of the Values
These exact constants were empirically measured and established in the highly cited 2001 paper by Ruifrok and Johnston (*"Quantification of histochemical staining by color deconvolution"*). The authors used optical density principles based on the Beer-Lambert law to determine the exact normalized vector paths these specific stains take in RGB color space.

#### Application in the Code
In the `enhance_he_deconvolution` function, `skimage.color.separate_stains` utilizes this matrix to:
1.  Convert the RGB image into Optical Density (OD) space using the formula $OD = -\log_{10}(I/I_0)$.
2.  Multiply the OD image by the inverse of the `hed_matrix` to isolate the specific concentrations of each stain.
3.  Extract `stains[:, :, 0]`, which explicitly targets the isolated Hematoxylin (nuclear) channel, making it easier for cell-segmentation models to identify nuclei without interference from Eosin or DAB.

In [ ]:
import cv2
import numpy as np
from skimage.color import separate_stains, hdx_from_rgb
from skimage.filters import unsharp_mask
import os
from tqdm.notebook import tqdm

class EnhancementPipeline:
    def __init__(self):
        self.clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(32,32)) # Best settings for HE & SAM
        self.hed_matrix = np.array([
            [0.650, 0.704, 0.286],  # Hematoxylin
            [0.072, 0.990, 0.105],  # Eosin
            [0.268, 0.570, 0.776]   # DAB
        ])

    def get_baseline(self, img):
        if len(img.shape) == 3:
            return cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        return img

    def enhance_he_clahe(self, img):
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l = self.clahe.apply(l)
        return cv2.cvtColor(cv2.merge((l,a,b)), cv2.COLOR_LAB2RGB)

    def enhance_he_deconvolution(self, img):
        stains = separate_stains(img, self.hed_matrix)
        h_channel = stains[:, :, 0]
        return cv2.normalize(h_channel, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    def enhance_mif_laplacian(self, gray_img):
        lap = cv2.Laplacian(gray_img, cv2.CV_64F)
        sharpened = np.clip(gray_img - 0.5 * lap, 0, 255).astype(np.uint8)
        return sharpened

    def enhance_mif_unsharp(self, gray_img):
        gauss = cv2.GaussianBlur(gray_img, (0, 0), 1.2) # Best settings for mIF & SAM
        return cv2.addWeighted(gray_img, 2.0, gauss, -0.5, 0)

class Evaluator:
    @staticmethod
    def calculate_binary_metrics(pred_mask, true_mask):
        pred_bool, true_bool = pred_mask > 0, true_mask > 0
        tp = np.sum(pred_bool & true_bool)
        fp = np.sum(pred_bool & ~true_bool)
        fn = np.sum(~pred_bool & true_bool)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1_dice = (2 * tp) / ((2 * tp) + fp + fn) if ((2 * tp) + fp + fn) > 0 else 0.0
        return {"Precision": precision, "Recall": recall, "F1_Score": f1_dice, "Dice": f1_dice}

    @staticmethod
    def calculate_panoptic_quality(pred_inst, true_inst, iou_threshold=0.5):
        true_ids = np.unique(true_inst)[1:]
        pred_ids = np.unique(pred_inst)[1:]

        tp_iou_sum, tp = 0, 0
        fp, fn = len(pred_ids), len(true_ids)

        for t_id in true_ids:
            t_mask = (true_inst == t_id)
            best_iou = 0
            for p_id in pred_ids:
                p_mask = (pred_inst == p_id)
                intersection = np.logical_and(t_mask, p_mask).sum()
                union = np.logical_or(t_mask, p_mask).sum()
                iou = intersection / union if union > 0 else 0
                if iou > best_iou:
                    best_iou = iou

            if best_iou >= iou_threshold:
                tp += 1
                fp -= 1
                fn -= 1
                tp_iou_sum += best_iou

        pq = tp_iou_sum / (tp + 0.5 * fp + 0.5 * fn) if (tp + 0.5 * fp + 0.5 * fn) > 0 else 0
        return {"Panoptic Quality": pq}

def get_cellbindb_paths(base_dir):
    image_paths, binary_mask_paths, instance_mask_paths = [], [], []
    if not os.path.exists(base_dir):
        return [], [], []
    subdirs = [os.path.join(base_dir, d) for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    for subdir in subdirs:
        folder_name = os.path.basename(subdir)
        img_path = os.path.join(subdir, f"{folder_name}-img.tif")
        bin_mask_path = os.path.join(subdir, f"{folder_name}-mask.tif")
        inst_mask_path = os.path.join(subdir, f"{folder_name}-instancemask.tif")
        if not os.path.exists(img_path):
            img_path += "f"
            bin_mask_path += "f"
            inst_mask_path += "f"
        if os.path.exists(img_path) and os.path.exists(bin_mask_path) and os.path.exists(inst_mask_path):
            image_paths.append(img_path)
            binary_mask_paths.append(bin_mask_path)
            instance_mask_paths.append(inst_mask_path)
    return image_paths, binary_mask_paths, instance_mask_paths

def get_completed_images(csv_path):
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            if 'Image' in df.columns:
                return set(df['Image'].unique())
        except pd.errors.EmptyDataError:
            return set()
    return set()

HE_BASE_DIR = BASE_DIR + "/data/raw_baseline/CellBinDB/HE"
MIF_BASE_DIR = BASE_DIR + "/data/raw_baseline/CellBinDB/mIF"

he_i, he_b, he_inst = get_cellbindb_paths(HE_BASE_DIR)
mif_i, mif_b, mif_inst = get_cellbindb_paths(MIF_BASE_DIR)

## Cellpose

In [ ]:
# @title
!pip install -q cellpose

import gc
import torch
import pandas as pd
import os
import cv2
from cellpose import models, core

class CellposeWrapper:
    def __init__(self):
        use_gpu = core.use_gpu()
        print(f"Cellpose using GPU: {use_gpu}")
        self.model = models.CellposeModel(gpu=use_gpu)

    def predict(self, image):
        masks, flows, styles = self.model.eval(image, diameter=None)
        return masks

def run_cellpose_experiment(img_paths, bin_paths, inst_paths, modality, output_csv):
    enhancer = EnhancementPipeline()
    evaluator = Evaluator()
    model = CellposeWrapper()

    processed_images = get_completed_images(output_csv)
    if processed_images:
        print(f"Resuming experiment. Found {len(processed_images)} already processed images.")

    remaining_indices = [
        i for i, path in enumerate(img_paths)
        if os.path.basename(path) not in processed_images
    ]

    already_done_count = len(img_paths) - len(remaining_indices)

    for idx in tqdm(remaining_indices,
                    desc=f"Processing {modality} Images",
                    initial=already_done_count,
                    total=len(img_paths)):

        img_path, bin_path, inst_path = img_paths[idx], bin_paths[idx], inst_paths[idx]
        image_name = os.path.basename(img_path)

        raw_img = cv2.imread(img_path)
        raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
        gt_bin_mask = cv2.imread(bin_path, cv2.IMREAD_UNCHANGED)
        gt_inst_mask = cv2.imread(inst_path, cv2.IMREAD_UNCHANGED)

        conditions = {"baseline": enhancer.get_baseline(raw_img)}
        if modality == "HE":
            conditions["clahe"] = enhancer.enhance_he_clahe(raw_img)
            conditions["deconv"] = enhancer.enhance_he_deconvolution(raw_img)
        elif modality == "mIF":
            gray_img = enhancer.get_baseline(raw_img)
            conditions["laplacian"] = enhancer.enhance_mif_laplacian(gray_img)
            conditions["unsharp"] = enhancer.enhance_mif_unsharp(gray_img)

        image_results = []
        for condition_name, processed_img in conditions.items():
            print(f"Cellpose | {modality} | Image {idx+1}/{len(img_paths)} | {condition_name}")
            pred_mask = model.predict(processed_img)
            bin_metrics = evaluator.calculate_binary_metrics(pred_mask, gt_bin_mask)
            pq_metric = evaluator.calculate_panoptic_quality(pred_mask, gt_inst_mask)
            image_results.append({
                "Image": image_name,
                "Model": "Cellpose",
                "Condition": condition_name,
                **bin_metrics,
                **pq_metric
            })

        df_batch = pd.DataFrame(image_results)
        write_header = not os.path.exists(output_csv)
        df_batch.to_csv(output_csv, mode='a', header=write_header, index=False)

        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    return pd.read_csv(output_csv) if os.path.exists(output_csv) else pd.DataFrame()

HE_CSV_PATH = BASE_DIR + "/results/Cellpose_HE_results.csv"
MIF_CSV_PATH = BASE_DIR + "/results/Cellpose_mIF_results.csv"

if he_i:
    print(f"Running Cellpose on H&E ({len(he_i)} images)...")
    run_cellpose_experiment(he_i, he_b, he_inst, "HE", HE_CSV_PATH)

if mif_i:
    print(f"Running Cellpose on mIF ({len(mif_i)} images)...")
    run_cellpose_experiment(mif_i, mif_b, mif_inst, "mIF", MIF_CSV_PATH)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 66.8 MB/s eta 0:00:00


## SAM

In [ ]:
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

import gc
import torch
import pandas as pd
import os
import cv2
import numpy as np
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

class SAMWrapper:
    def __init__(self, checkpoint_path="sam_vit_h_4b8939.pth", model_type="vit_h"):
        print("Loading SAM weights into VRAM...")
        sam = sam_model_registry[model_type](checkpoint=checkpoint_path)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        sam.to(device=device)
        # Automatically generates a set of segmentation masks for an input image by densely prompting the SAM model
        # and filtering/merging the resulting masks to produce high-quality object region proposals without manual prompts
        self.mask_generator = SamAutomaticMaskGenerator(sam)

    def predict(self, image):
        if len(image.shape) == 2:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        sam_results = self.mask_generator.generate(image)
        instance_mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.int32)
        sorted_results = sorted(sam_results, key=(lambda x: x['area']), reverse=True)
        for idx, ann in enumerate(sorted_results):
            instance_mask[ann['segmentation']] = idx + 1
        return instance_mask

def run_sam_experiment(img_paths, bin_paths, inst_paths, modality, output_csv):
    enhancer = EnhancementPipeline()
    evaluator = Evaluator()
    model = SAMWrapper()

    processed_images = get_completed_images(output_csv)
    if processed_images:
        print(f"Resuming experiment. Found {len(processed_images)} already processed images.")

    remaining_indices = [
        i for i, path in enumerate(img_paths)
        if os.path.basename(path) not in processed_images
    ]

    already_done_count = len(img_paths) - len(remaining_indices)

    for idx in tqdm(remaining_indices,
                    desc=f"Processing {modality} Images",
                    initial=already_done_count,
                    total=len(img_paths)):

        img_path, bin_path, inst_path = img_paths[idx], bin_paths[idx], inst_paths[idx]
        image_name = os.path.basename(img_path)

        raw_img = cv2.imread(img_path)
        raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
        gt_bin_mask = cv2.imread(bin_path, cv2.IMREAD_UNCHANGED)
        gt_inst_mask = cv2.imread(inst_path, cv2.IMREAD_UNCHANGED)

        conditions = {"baseline": enhancer.get_baseline(raw_img)}
        if modality == "HE":
            conditions["clahe"] = enhancer.enhance_he_clahe(raw_img)
            conditions["deconv"] = enhancer.enhance_he_deconvolution(raw_img)
        elif modality == "mIF":
            gray_img = enhancer.get_baseline(raw_img)
            conditions["laplacian"] = enhancer.enhance_mif_laplacian(gray_img)
            conditions["unsharp"] = enhancer.enhance_mif_unsharp(gray_img)

        image_results = []
        for condition_name, processed_img in conditions.items():
            print(f"SAM | {modality} | Image {idx+1}/{len(img_paths)} | {condition_name}")
            pred_mask = model.predict(processed_img)
            bin_metrics = evaluator.calculate_binary_metrics(pred_mask, gt_bin_mask)
            pq_metric = evaluator.calculate_panoptic_quality(pred_mask, gt_inst_mask)
            image_results.append({
                "Image": image_name,
                "Model": "SAM",
                "Condition": condition_name,
                **bin_metrics,
                **pq_metric
            })

        df_batch = pd.DataFrame(image_results)
        write_header = not os.path.exists(output_csv)
        df_batch.to_csv(output_csv, mode='a', header=write_header, index=False)

        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    return pd.read_csv(output_csv) if os.path.exists(output_csv) else pd.DataFrame()

HE_CSV_PATH = BASE_DIR + "/results/SAM_HE_results.csv"
MIF_CSV_PATH = BASE_DIR + "/results/SAM_mIF_results.csv"

if he_i:
    print(f"Running SAM on H&E ({len(he_i)} images)...")
    run_sam_experiment(he_i, he_b, he_inst, "HE", HE_CSV_PATH)

if mif_i:
    print(f"Running SAM on mIF ({len(mif_i)} images)...")
    run_sam_experiment(mif_i, mif_b, mif_inst, "mIF", MIF_CSV_PATH)

  Preparing metadata (setup.py) ... done
Running SAM on H&E (305 images)...
Loading SAM weights into VRAM...


Processing HE Images:   0%|          | 0/305 [00:00<?, ?it/s]

SAM | HE | Image 1/305 | baseline
SAM | HE | Image 1/305 | clahe
SAM | HE | Image 1/305 | deconv
SAM | HE | Image 2/305 | baseline
SAM | HE | Image 2/305 | clahe
SAM | HE | Image 2/305 | deconv
SAM | HE | Image 3/305 | baseline
SAM | HE | Image 3/305 | clahe
SAM | HE | Image 3/305 | deconv
SAM | HE | Image 4/305 | baseline
SAM | HE | Image 4/305 | clahe
SAM | HE | Image 4/305 | deconv
SAM | HE | Image 5/305 | baseline
SAM | HE | Image 5/305 | clahe
SAM | HE | Image 5/305 | deconv
SAM | HE | Image 6/305 | baseline
SAM | HE | Image 6/305 | clahe
SAM | HE | Image 6/305 | deconv
SAM | HE | Image 7/305 | baseline
SAM | HE | Image 7/305 | clahe
SAM | HE | Image 7/305 | deconv
SAM | HE | Image 8/305 | baseline
SAM | HE | Image 8/305 | clahe
SAM | HE | Image 8/305 | deconv
SAM | HE | Image 9/305 | baseline
SAM | HE | Image 9/305 | clahe
SAM | HE | Image 9/305 | deconv
SAM | HE | Image 10/305 | baseline
SAM | HE | Image 10/305 | clahe
SAM | HE | Image 10/305 | deconv
SAM | HE | Image 11/305 | b

## DeepCell

In [ ]:
%%bash
add-apt-repository ppa:deadsnakes/ppa -y > /dev/null 2>&1
apt-get update > /dev/null 2>&1
apt-get install python3.10 python3.10-distutils -y > /dev/null 2>&1
curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10 > /dev/null 2>&1

wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
dpkg -i cuda-keyring_1.1-1_all.deb > /dev/null 2>&1
apt-get update > /dev/null 2>&1
apt-get install -y cuda-toolkit-11-8 libcudnn8 > /dev/null 2>&1

python3.10 -m pip install -q deepcell opencv-python-headless scikit-image pandas > /dev/null 2>&1

In [ ]:
%%writefile run_deepcell.py
import os
import sys

os.environ['LD_LIBRARY_PATH'] = f"/usr/local/cuda-11.8/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"
os.environ["DEEPCELL_ACCESS_TOKEN"] = "K8XjkycL.dxfcOwsm7FFMR6JYTApMkYe5Eo27UDc7"

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import gc
from deepcell.applications import Mesmer
from skimage.color import separate_stains, hdx_from_rgb
from skimage.filters import unsharp_mask
from tqdm.notebook import tqdm

def get_completed_images(csv_path):
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            if 'Image' in df.columns:
                return set(df['Image'].unique())
        except pd.errors.EmptyDataError:
            return set()
    return set()

class EnhancementPipeline:
    def __init__(self):
        self.clahe = cv2.createCLAHE(clipLimit=0.5, tileGridSize=(12,12)) # Best settings for HE & DeepCell
        self.hed_matrix = np.array([[0.650, 0.704, 0.286], [0.072, 0.990, 0.105], [0.268, 0.570, 0.776]])
    def get_baseline(self, img):
        if len(img.shape) == 3: return cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        return img
    def enhance_he_clahe(self, img):
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l = self.clahe.apply(l)
        return cv2.cvtColor(cv2.merge((l,a,b)), cv2.COLOR_LAB2RGB)
    def enhance_he_deconvolution(self, img):
        stains = separate_stains(img, self.hed_matrix)
        return cv2.normalize(stains[:, :, 0], None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    def enhance_mif_laplacian(self, gray_img):
        lap = cv2.Laplacian(gray_img, cv2.CV_64F)
        return np.clip(gray_img - 0.5 * lap, 0, 255).astype(np.uint8)
    def enhance_mif_unsharp(self, gray_img):
        gauss = cv2.GaussianBlur(gray_img, (0, 0), 0.8) # Best settings for mIF & DeepCell
        return cv2.addWeighted(gray_img, 3.0, gauss, -2.0, 0)

class Evaluator:
    @staticmethod
    def calculate_binary_metrics(pred_mask, true_mask):
        pred_bool, true_bool = pred_mask > 0, true_mask > 0
        tp, fp, fn = np.sum(pred_bool & true_bool), np.sum(pred_bool & ~true_bool), np.sum(~pred_bool & true_bool)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1_dice = (2 * tp) / ((2 * tp) + fp + fn) if ((2 * tp) + fp + fn) > 0 else 0.0
        return {"Precision": precision, "Recall": recall, "F1_Score": f1_dice, "Dice": f1_dice}
    @staticmethod
    def calculate_panoptic_quality(pred_inst, true_inst, iou_threshold=0.5):
        true_ids, pred_ids = np.unique(true_inst)[1:], np.unique(pred_inst)[1:]
        tp_iou_sum, tp, fp, fn = 0, 0, len(pred_ids), len(true_ids)
        for t_id in true_ids:
            t_mask = (true_inst == t_id)
            best_iou = max([np.logical_and(t_mask, (pred_inst == p_id)).sum() / np.logical_or(t_mask, (pred_inst == p_id)).sum() if np.logical_or(t_mask, (pred_inst == p_id)).sum() > 0 else 0 for p_id in pred_ids] + [0])
            if best_iou >= iou_threshold: tp += 1; fp -= 1; fn -= 1; tp_iou_sum += best_iou
        return {"Panoptic Quality": tp_iou_sum / (tp + 0.5 * fp + 0.5 * fn) if (tp + 0.5 * fp + 0.5 * fn) > 0 else 0}

class DeepCellWrapper:
    def __init__(self):
        print("Loading DeepCell Mesmer weights...")
        self.model = Mesmer()

    def predict(self, image):
        if len(image.shape) == 3 and image.shape[-1] == 3:
            image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

        blank_channel = np.zeros_like(image)
        image_2ch = np.stack((image, blank_channel), axis=-1)
        img_expanded = np.expand_dims(image_2ch, axis=0)

        prediction = self.model.predict(img_expanded, image_mpp=0.5, compartment='nuclear')

        return prediction[0, ..., 0]

def get_cellbindb_paths(base_dir):
    image_paths, binary_mask_paths, instance_mask_paths = [], [], []
    subdirs = [os.path.join(base_dir, d) for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    for subdir in subdirs:
        folder_name = os.path.basename(subdir)
        img_path, bin_mask_path, inst_mask_path = os.path.join(subdir, f"{folder_name}-img.tif"), os.path.join(subdir, f"{folder_name}-mask.tif"), os.path.join(subdir, f"{folder_name}-instancemask.tif")
        if not os.path.exists(img_path): img_path += "f"; bin_mask_path += "f"; inst_mask_path += "f"
        if os.path.exists(img_path) and os.path.exists(bin_mask_path) and os.path.exists(inst_mask_path):
            image_paths.append(img_path); binary_mask_paths.append(bin_mask_path); instance_mask_paths.append(inst_mask_path)
    return image_paths, binary_mask_paths, instance_mask_paths

def run_modality(i_paths, b_paths, inst_paths, modality, out_csv):
    if not i_paths: return
    enhancer, evaluator, model = EnhancementPipeline(), Evaluator(), DeepCellWrapper()

    processed_images = get_completed_images(out_csv)
    if processed_images:
        print(f"Resuming DeepCell {modality} experiment. Found {len(processed_images)} already processed images.")

    remaining_indices = [
        i for i, path in enumerate(i_paths)
        if os.path.basename(path) not in processed_images
    ]

    already_done_count = len(i_paths) - len(remaining_indices)

    for idx in tqdm(remaining_indices,
                    desc=f"Processing {modality} Images",
                    initial=already_done_count,
                    total=len(i_paths)):

        image_name = os.path.basename(i_paths[idx])

        raw_img = cv2.cvtColor(cv2.imread(i_paths[idx]), cv2.COLOR_BGR2RGB)
        gt_bin_mask = cv2.imread(b_paths[idx], cv2.IMREAD_UNCHANGED)
        gt_inst_mask = cv2.imread(inst_paths[idx], cv2.IMREAD_UNCHANGED)

        conditions = {"baseline": enhancer.get_baseline(raw_img)}
        if modality == "HE":
            conditions["clahe"] = enhancer.enhance_he_clahe(raw_img)
            conditions["deconv"] = enhancer.enhance_he_deconvolution(raw_img)
        elif modality == "mIF":
            gray_img = enhancer.get_baseline(raw_img)
            conditions["laplacian"] = enhancer.enhance_mif_laplacian(gray_img)
            conditions["unsharp"] = enhancer.enhance_mif_unsharp(gray_img)

        image_results = []
        for condition_name, processed_img in conditions.items():
            print(f"DeepCell | {modality} | Image {idx+1}/{len(i_paths)} | {condition_name}")
            pred_mask = model.predict(processed_img)

            bin_metrics = evaluator.calculate_binary_metrics(pred_mask, gt_bin_mask)
            pq_metric = evaluator.calculate_panoptic_quality(pred_mask, gt_inst_mask)

            image_results.append({
                "Image": image_name,
                "Model": "DeepCell",
                "Condition": condition_name,
                **bin_metrics,
                **pq_metric
            })

        df_batch = pd.DataFrame(image_results)
        write_header = not os.path.exists(out_csv)
        df_batch.to_csv(out_csv, mode='a', header=write_header, index=False)

        gc.collect()
        tf.keras.backend.clear_session()

def main():
    if len(sys.argv) < 2:
        raise ValueError(
            "Error: BASE_DIR must be supplied as the first command-line argument.\n"
            "Usage: python run_deepcell.py /path/to/BASE_DIR"
        )

    BASE_DIR = sys.argv[1]


    HE_BASE_DIR = BASE_DIR + "/data/raw_baseline/CellBinDB/HE"
    MIF_BASE_DIR = BASE_DIR + "/data/raw_baseline/CellBinDB/mIF"

    i_paths, b_paths, inst_paths = get_cellbindb_paths(HE_BASE_DIR)
    run_modality(i_paths, b_paths, inst_paths, "HE", BASE_DIR + "/results/DeepCell_HE_results.csv")

    i_paths, b_paths, inst_paths = get_cellbindb_paths(MIF_BASE_DIR)
    run_modality(i_paths, b_paths, inst_paths, "mIF", BASE_DIR + "/results/DeepCell_mIF_results.csv")

if __name__ == "__main__":
    main()

Writing run_deepcell.py


In [ ]:
!python3.10 run_deepcell.py "{BASE_DIR}"

Loading DeepCell Mesmer weights...
INFO:root:Checking for cached data
INFO:root:Making request to server
INFO:root:Downloading models/MultiplexSegmentation-9.tar.gz with size 92.4 MB to /root/.deepcell/models
92.4MB [00:05, 16.5MB/s]
INFO:root:🎉 Successfully downloaded file to /root/.deepcell/models/MultiplexSegmentation-9.tar.gz
INFO:root:Extracting /root/.deepcell/models/MultiplexSegmentation-9.tar.gz
INFO:root:Successfully extracted /root/.deepcell/models/MultiplexSegmentation-9.tar.gz into /root/.deepcell/models
2026-04-01 07:02:58.899723: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-04-01 07:03:00.529944: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-04-01 07:03:00.532950: I tensorfl

## HoVer-Net

In [ ]:
!pip install -q typeguard gdown
import os
import cv2
import torch
import numpy as np
import pandas as pd
import gc
import sys
import importlib.util
from tqdm.notebook import tqdm

os.makedirs('weights', exist_ok=True)
if not os.path.exists('weights/hovernet_consep.tar'):
    print("Downloading HoVer-Net weights...")
    !gdown --id 1KntZge40tAHgyXmHYVqZZ5d2p_4Qr2l5 -O weights/hovernet_consep.tar

if not os.path.exists('hover_net'):
    !git clone https://github.com/vqdang/hover_net.git

target_file = os.path.abspath('hover_net/models/hovernet/net_desc.py')

if os.path.exists(target_file):
    hv_dir = os.path.dirname(target_file)
    hv_root = os.path.abspath('hover_net')

    if hv_root not in sys.path: sys.path.insert(0, hv_root)

    import importlib.util

    def load_as_module(name, path):
        spec = importlib.util.spec_from_file_location(name, path)
        mod = importlib.util.module_from_spec(spec)
        sys.modules[name] = mod
        spec.loader.exec_module(mod)
        return mod

    try:
        load_as_module("config", os.path.join(hv_root, "config.py"))
        load_as_module("models.hovernet.utils", os.path.join(hv_dir, "utils.py"))
        load_as_module("models.hovernet.net_utils", os.path.join(hv_dir, "net_utils.py"))

        spec = importlib.util.spec_from_file_location("models.hovernet.net_desc", target_file)
        hv_module = importlib.util.module_from_spec(spec)
        hv_module.__package__ = "models.hovernet"
        sys.modules["models.hovernet.net_desc"] = hv_module
        spec.loader.exec_module(hv_module)

        HoVerNet = hv_module.HoVerNet
        print("HoVerNet class imported successfully.")
    except Exception as e:
        print(f"Import error: {e}")
else:
    raise FileNotFoundError(f"Could not locate net_desc.py at {target_file}")

def get_completed_images(csv_path):
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            return set(df['Image'].unique()) if 'Image' in df.columns else set()
        except: return set()
    return set()

class HoVerNetWrapper:
    def __init__(self, checkpoint_path="weights/hovernet_consep.tar"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = HoVerNet()
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        state_dict = checkpoint['desc'] if 'desc' in checkpoint else checkpoint
        self.model.load_state_dict(state_dict, strict=False)
        self.model.to(self.device).eval()
        print("HoVer-Net weights loaded.")

    def predict(self, image):
        if len(image.shape) == 2: image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        h, w = image.shape[:2]
        image_resized = cv2.resize(image, (270, 270), interpolation=cv2.INTER_LINEAR)
        img_tensor = torch.from_numpy(image_resized).permute(2, 0, 1).float().unsqueeze(0) / 255.0
        with torch.no_grad():
            pred_dict = self.model(img_tensor.to(self.device))
            pred_np = pred_dict['np'].squeeze().cpu().numpy()
            pred_mask = (pred_np[..., 1] > 0.5).astype(np.uint8)
            pred_mask = cv2.resize(pred_mask, (w, h), interpolation=cv2.INTER_NEAREST)
        return pred_mask.astype(np.int32)

def run_hovernet_experiment(img_paths, bin_paths, inst_paths, modality, output_csv):
    enhancer, evaluator, model = EnhancementPipeline(), Evaluator(), HoVerNetWrapper()
    processed = get_completed_images(output_csv)
    rem_idx = [i for i, p in enumerate(img_paths) if os.path.basename(p) not in processed]

    for idx in tqdm(rem_idx, desc=f"HoVer-Net {modality}", initial=len(img_paths)-len(rem_idx), total=len(img_paths)):
        img_p, bin_p, inst_p = img_paths[idx], bin_paths[idx], inst_paths[idx]
        raw = cv2.cvtColor(cv2.imread(img_p), cv2.COLOR_BGR2RGB)
        gt_bin, gt_inst = cv2.imread(bin_p, cv2.IMREAD_UNCHANGED), cv2.imread(inst_p, cv2.IMREAD_UNCHANGED)

        conds = {"baseline": enhancer.get_baseline(raw)}
        if modality == "HE":
            conds["clahe"] = enhancer.enhance_he_clahe(raw)
            conds["deconv"] = enhancer.enhance_he_deconvolution(raw)
        else:
            gray = enhancer.get_baseline(raw)
            conds["laplacian"], conds["unsharp"] = enhancer.enhance_mif_laplacian(gray), enhancer.enhance_mif_unsharp(gray)

        res = []
        for name, proc_img in conds.items():
            print(f"HoVerNet | {modality} | Image {idx+1}/{len(img_paths)} | {name}")
            pred = model.predict(proc_img)
            res.append({"Image": os.path.basename(img_p), "Model": "HoVer-Net", "Condition": name,
                        **evaluator.calculate_binary_metrics(pred, gt_bin), **evaluator.calculate_panoptic_quality(pred, gt_inst)})

        pd.DataFrame(res).to_csv(output_csv, mode='a', header=not os.path.exists(output_csv), index=False)
        gc.collect(); torch.cuda.empty_cache()

if 'he_i' in locals() and he_i:
    print(f"Starting HoVer-Net on H&E ({len(he_i)} images)...")
    run_hovernet_experiment(he_i, he_b, he_inst, "HE", os.path.join(BASE_DIR, "results/HoVerNet_HE_results.csv"))

if 'mif_i' in locals() and mif_i:
    print(f"Starting HoVer-Net on mIF ({len(mif_i)} images)...")
    run_hovernet_experiment(mif_i, mif_b, mif_inst, "mIF", os.path.join(BASE_DIR, "results/HoVerNet_mIF_results.csv"))

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1KntZge40tAHgyXmHYVqZZ5d2p_4Qr2l5
To: /content/weights/hovernet_consep.tar
100% 94.2M/94.2M [00:01<00:00, 48.5MB/s]
Cloning into 'hover_net'...
remote: Enumerating objects: 2032, done.
remote: Counting objects: 100% (494/494), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 2032 (delta 424), reused 399 (delta 399), pack-reused 1538 (from 2)
Receiving objects: 100% (2032/2032), 40.39 MiB | 16.56 MiB/s, done.
Resolving deltas: 100% (1249/1249), done.
HoVerNet class imported successfully.
Starting HoVer-Net on H&E (305 images)...
HoVer-Net weights loaded.


HoVer-Net HE:   3%|3         | 10/305 [00:00<?, ?it/s]

HoVerNet | HE | Image 11/305 | baseline
HoVerNet | HE | Image 11/305 | clahe
HoVerNet | HE | Image 11/305 | deconv
HoVerNet | HE | Image 12/305 | baseline
HoVerNet | HE | Image 12/305 | clahe
HoVerNet | HE | Image 12/305 | deconv
HoVerNet | HE | Image 13/305 | baseline
HoVerNet | HE | Image 13/305 | clahe
HoVerNet | HE | Image 13/305 | deconv
HoVerNet | HE | Image 14/305 | baseline
HoVerNet | HE | Image 14/305 | clahe
HoVerNet | HE | Image 14/305 | deconv
HoVerNet | HE | Image 15/305 | baseline
HoVerNet | HE | Image 15/305 | clahe
HoVerNet | HE | Image 15/305 | deconv
HoVerNet | HE | Image 16/305 | baseline
HoVerNet | HE | Image 16/305 | clahe
HoVerNet | HE | Image 16/305 | deconv
HoVerNet | HE | Image 17/305 | baseline
HoVerNet | HE | Image 17/305 | clahe
HoVerNet | HE | Image 17/305 | deconv
HoVerNet | HE | Image 18/305 | baseline
HoVerNet | HE | Image 18/305 | clahe
HoVerNet | HE | Image 18/305 | deconv
HoVerNet | HE | Image 19/305 | baseline
HoVerNet | HE | Image 19/305 | clahe
HoV

HoVer-Net mIF:   0%|          | 0/60 [00:00<?, ?it/s]

HoVerNet | mIF | Image 1/60 | baseline
HoVerNet | mIF | Image 1/60 | laplacian
HoVerNet | mIF | Image 1/60 | unsharp
HoVerNet | mIF | Image 2/60 | baseline
HoVerNet | mIF | Image 2/60 | laplacian
HoVerNet | mIF | Image 2/60 | unsharp
HoVerNet | mIF | Image 3/60 | baseline
HoVerNet | mIF | Image 3/60 | laplacian
HoVerNet | mIF | Image 3/60 | unsharp
HoVerNet | mIF | Image 4/60 | baseline
HoVerNet | mIF | Image 4/60 | laplacian
HoVerNet | mIF | Image 4/60 | unsharp
HoVerNet | mIF | Image 5/60 | baseline
HoVerNet | mIF | Image 5/60 | laplacian
HoVerNet | mIF | Image 5/60 | unsharp
HoVerNet | mIF | Image 6/60 | baseline
HoVerNet | mIF | Image 6/60 | laplacian
HoVerNet | mIF | Image 6/60 | unsharp
HoVerNet | mIF | Image 7/60 | baseline
HoVerNet | mIF | Image 7/60 | laplacian
HoVerNet | mIF | Image 7/60 | unsharp
HoVerNet | mIF | Image 8/60 | baseline
HoVerNet | mIF | Image 8/60 | laplacian
HoVerNet | mIF | Image 8/60 | unsharp
HoVerNet | mIF | Image 9/60 | baseline
HoVerNet | mIF | Image 9/